# Milestone 4: Live Demo

Runs the integrated pipeline on held-out test images or on an image you provide: the Milestone 3 detector finds
the faces and the Milestone 1 classifier (adapted to face crops) names each one.

* **Models and settings:** `results/milestone3/sweep/best_model.pt`, `checkpoints/efficientnet_b0_crops.pt` and
  `results/milestone4/pipeline_settings.json` (from `Integrated_Pipeline.ipynb`).
* Each face is boxed in its identity color with the name and confidence. `?` marks a face where the detector and
  the classifier disagree; the table shows both answers.

#### 0. Environment setup (local Jupyter or Google Colab)
Same cell as the other notebooks: on Colab it mounts Drive and installs the requirements, locally it finds the repo root.

In [1]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/rasmussenj03/IE7615-Group3-Project1.git"
COLAB_USE_DRIVE = True  # False -> clone into /content (outputs are lost when the runtime ends)
COLAB_REPO_DIR = ("/content/drive/MyDrive/IE7615-Group3-Project1" if COLAB_USE_DRIVE
                  else "/content/IE7615-Group3-Project1")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if COLAB_USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    if not Path(COLAB_REPO_DIR, "src").is_dir():
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        clone = subprocess.run(["git", "clone", "-q", REPO_URL.replace("https://", f"https://{token}@"),
                                COLAB_REPO_DIR], capture_output=True)
        if clone.returncode != 0:
            raise RuntimeError("git clone failed - check the GITHUB_TOKEN secret and repo access")
        subprocess.run(["git", "-C", COLAB_REPO_DIR, "remote", "set-url", "origin", REPO_URL])
    os.chdir(COLAB_REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    root = Path.cwd()
    while not (root / "src" / "pipeline.py").exists() and root != root.parent:
        root = root.parent
    os.chdir(root)

PROJECT_ROOT = Path.cwd()

#### 1. Load the pipeline

In [2]:
import pandas as pd

from src.pipeline import CelebrityPipeline

pipe = CelebrityPipeline()
TEST_DIR = PROJECT_ROOT / "data" / "detection" / "images" / "test"
print("Settings:", pipe.settings)


def identify(image, conf=None):
    """Runs the pipeline on one image; returns the annotated image and one row per face."""
    faces = pipe.predict(image, conf)
    table = pd.DataFrame(faces, columns=["name", "confidence", "detector", "classifier", "agree", "box"])
    return pipe.annotate(image, faces), table

Settings: {'det_conf': 0.7, 'nms_iou': 0.7, 'imgsz': 640, 'identity': 'detector'}


#### 2. Launch the app
Pick a test image, drag and drop or upload an image, or take a photo with the webcam; the results update when the
image or the confidence threshold changes. Run the next cell to stop the app.

In [3]:
import gradio as gr

TEST_IMAGES = sorted(p.name for p in TEST_DIR.glob("*.jpg"))


def run(image, conf):
    """Pipeline for the app; clears the outputs when no image is selected."""
    return identify(image, conf) if image else (None, None)


with gr.Blocks(title="IE7615 Group 3 demo") as app:
    gr.Markdown("## IE7615 Group 3: celebrity detection (YOLOv8n) and identification (EfficientNet-B0)\n"
                "Pick a test image, drag and drop or upload an image, or take a webcam photo.")
    with gr.Row():
        with gr.Column():
            pick = gr.Dropdown(TEST_IMAGES, value="test_0001.jpg", label="Test image")
            image = gr.Image(str(TEST_DIR / "test_0001.jpg"), type="filepath", sources=["upload", "webcam"],
                             label="Input image (drag and drop, upload or webcam)")
            conf = gr.Slider(0.05, 0.95, value=pipe.settings["det_conf"], step=0.05, label="Detector confidence")
            clear = gr.ClearButton(value="Clear")
        with gr.Column():
            result = gr.Image(label="Detected and identified faces")
            faces = gr.Dataframe(label="Faces")

    clear.add([pick, image, result, faces])
    pick.input(lambda name: str(TEST_DIR / name), pick, image)
    image.change(run, [image, conf], [result, faces])
    conf.release(run, [image, conf], [result, faces])
    app.load(run, [image, conf], [result, faces])

app.launch(share=IN_COLAB)

/Users/pmodi/Desktop/P2/Rhea/NU Grad/IE 7615/ie7615-env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [4]:
app.close()  # stops the app

Closing server running on port: 7860
